In [1]:
import os

DATA_ROOT = "/kaggle/input/datasets/startrek001/neuroshield-mslesseg-processed"

print("Dataset exists:", os.path.exists(DATA_ROOT))
print("\nContents:")

for item in sorted(os.listdir(DATA_ROOT)):
    print(" -", item)

Dataset exists: True

Contents:
 - mslesseg_manifest.csv
 - mslesseg_processed
 - splits


In [2]:
import os
import pandas as pd

SPLIT_ROOT = os.path.join(DATA_ROOT, "splits")

for name in ["train.csv", "validation.csv", "test.csv", "splits.csv"]:
    path = os.path.join(SPLIT_ROOT, name)

    print(f"\n{name}")
    print("Exists:", os.path.exists(path))

    df = pd.read_csv(path)
    print("Rows:", len(df))
    print("Columns:", list(df.columns))


train.csv
Exists: True
Rows: 71
Columns: ['dir', 'patient', 'timepoint', 't1_shape', 't2_shape', 'flair_shape', 'mask_shape', 'issues', 'ok', 'split']

validation.csv
Exists: True
Rows: 22
Columns: ['dir', 'patient', 'timepoint', 't1_shape', 't2_shape', 'flair_shape', 'mask_shape', 'issues', 'ok', 'split']

test.csv
Exists: True
Rows: 22
Columns: ['dir', 'patient', 'timepoint', 't1_shape', 't2_shape', 'flair_shape', 'mask_shape', 'issues', 'ok', 'split']

splits.csv
Exists: True
Rows: 115
Columns: ['dir', 'patient', 'timepoint', 't1_shape', 't2_shape', 'flair_shape', 'mask_shape', 'issues', 'ok', 'split']


In [3]:
import pandas as pd

train_df = pd.read_csv(os.path.join(SPLIT_ROOT, "train.csv"))
val_df = pd.read_csv(os.path.join(SPLIT_ROOT, "validation.csv"))
test_df = pd.read_csv(os.path.join(SPLIT_ROOT, "test.csv"))

train_patients = set(train_df["patient"])
val_patients = set(val_df["patient"])
test_patients = set(test_df["patient"])

print("Train patients:", len(train_patients))
print("Validation patients:", len(val_patients))
print("Test patients:", len(test_patients))

print("\nTrain/Validation overlap:", sorted(train_patients & val_patients))
print("Train/Test overlap:", sorted(train_patients & test_patients))
print("Validation/Test overlap:", sorted(val_patients & test_patients))

Train patients: 42
Validation patients: 11
Test patients: 22

Train/Validation overlap: []
Train/Test overlap: []
Validation/Test overlap: []


In [4]:
import os

PROCESSED_ROOT = os.path.join(DATA_ROOT, "mslesseg_processed")

print("Top level:")
for item in sorted(os.listdir(PROCESSED_ROOT)):
    print(" -", item)

PATIENT_ROOT = os.path.join(PROCESSED_ROOT, "mslesseg")

print("\nPatients:")
for patient in sorted(os.listdir(PATIENT_ROOT))[:5]:
    print(" -", patient)

    patient_path = os.path.join(PATIENT_ROOT, patient)

    if os.path.isdir(patient_path):
        print("   contents:", sorted(os.listdir(patient_path))[:10])

Top level:
 - mslesseg

Patients:
 - P1
   contents: ['T1', 'T2', 'T3']
 - P10
   contents: ['T1', 'T2']
 - P11
   contents: ['T1', 'T2']
 - P12
   contents: ['T1', 'T2', 'T3', 'T4']
 - P13
   contents: ['T1', 'T2']


In [5]:
import os
import numpy as np

PATIENT_ROOT = os.path.join(
    DATA_ROOT,
    "mslesseg_processed",
    "mslesseg"
)

# Find the first actual patient/timepoint directory
sample_dir = None

for patient in sorted(os.listdir(PATIENT_ROOT)):
    patient_path = os.path.join(PATIENT_ROOT, patient)

    if not os.path.isdir(patient_path):
        continue

    for timepoint in sorted(os.listdir(patient_path)):
        timepoint_path = os.path.join(patient_path, timepoint)

        if os.path.isdir(timepoint_path):
            sample_dir = timepoint_path
            break

    if sample_dir is not None:
        break

print("Sample directory:")
print(sample_dir)

print("\nVolume checks:")

for modality in ["T1", "T2", "FLAIR", "MASK"]:
    path = os.path.join(sample_dir, f"{modality}.npy")

    print(f"\n{modality}")
    print("Path exists:", os.path.exists(path))

    arr = np.load(path)

    print("Shape:", arr.shape)
    print("Dtype:", arr.dtype)
    print("Min:", arr.min())
    print("Max:", arr.max())
    print("NaN:", np.isnan(arr).any())

    if modality == "MASK":
        print("Unique values:", np.unique(arr))

Sample directory:
/kaggle/input/datasets/startrek001/neuroshield-mslesseg-processed/mslesseg_processed/mslesseg/P1/T1

Volume checks:

T1
Path exists: True
Shape: (182, 218, 182)
Dtype: float32
Min: -2.6910236
Max: 6.4663725
NaN: False

T2
Path exists: True
Shape: (182, 218, 182)
Dtype: float32
Min: -2.462574
Max: 6.083349
NaN: False

FLAIR
Path exists: True
Shape: (182, 218, 182)
Dtype: float32
Min: -3.5449905
Max: 3.6318762
NaN: False

MASK
Path exists: True
Shape: (182, 218, 182)
Dtype: uint8
Min: 0
Max: 1
NaN: False
Unique values: [0 1]


In [6]:
pip install monai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [7]:
"""
Model A -- reproduction of the Shifts-2.0 Challenge 3D U-Net baseline
(Malinin et al., NeurIPS 2022), retrained on MSLesSeg.

Honesty note (do not remove): this is NOT the original Shifts-2.0 checkpoint.
That checkpoint was trained on MSSEG-1+ISBI with 2 input channels (FLAIR+T1)
and cannot be reused directly on MSLesSeg's 3-channel (T1+T2+FLAIR) data.
This module reproduces the published architecture family (3D U-Net,
Ronneberger/Cicek-style) and training approach (Focal-Dice loss), retrained
from scratch on MSLesSeg. Ensemble size is reduced from the published 5
members to 1, due to GPU budget -- report this explicitly in any write-up.

Reference: https://github.com/Shifts-Project/shifts/tree/main/mswml
"""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.networks.nets import UNet
from monai.networks.layers import Norm


def build_model_a(in_channels: int = 3, out_channels: int = 1) -> nn.Module:
    """
    3D U-Net sized for MSLesSeg patches (default 3-channel: T1+T2+FLAIR).
    """
    return UNet(
        spatial_dims=3,
        in_channels=in_channels,
        out_channels=out_channels,
        channels=(32, 64, 128, 256, 320),
        strides=(2, 2, 2, 2),
        num_res_units=2,
        norm=Norm.INSTANCE,
        dropout=0.2,  # also enables MC Dropout later at inference time
    )


class FocalDiceLoss(nn.Module):
    """
    Focal loss + Dice loss, matching the Shifts-2.0 baseline's loss choice.
    """

    def __init__(self, focal_gamma: float = 2.0, focal_weight: float = 0.5,
                 smooth: float = 1e-5):
        super().__init__()
        self.focal_gamma = focal_gamma
        self.focal_weight = focal_weight
        self.smooth = smooth

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        # logits, target: [B, 1, D, H, W]
        probs = torch.sigmoid(logits)

        bce = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
        p_t = probs * target + (1 - probs) * (1 - target)
        focal = ((1 - p_t) ** self.focal_gamma * bce).mean()

        probs_flat = probs.reshape(probs.shape[0], -1)
        target_flat = target.reshape(target.shape[0], -1)
        intersection = (probs_flat * target_flat).sum(dim=1)
        dice_score_ = (2 * intersection + self.smooth) / (
            probs_flat.sum(dim=1) + target_flat.sum(dim=1) + self.smooth
        )
        dice_loss = 1 - dice_score_.mean()

        return self.focal_weight * focal + (1 - self.focal_weight) * dice_loss


@torch.no_grad()
def dice_score(logits: torch.Tensor, target: torch.Tensor, threshold: float = 0.5,
               smooth: float = 1e-5) -> float:
    """Batch-mean Dice for monitoring during training (not the final metric)."""
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    preds_flat = preds.reshape(preds.shape[0], -1)
    target_flat = target.reshape(target.shape[0], -1)
    intersection = (preds_flat * target_flat).sum(dim=1)
    dice = (2 * intersection + smooth) / (
        preds_flat.sum(dim=1) + target_flat.sum(dim=1) + smooth
    )
    return dice.mean().item()

In [8]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset


MODALITIES = ["T1", "T2", "FLAIR"]
PATCH_SIZE = (112, 160, 128)


class MSLesSegDataset(Dataset):
    """
    PyTorch dataset for MSLesSeg.

    Each sample contains:
        image: [3, D, H, W]
        mask:  [1, D, H, W]

    During training, a random patch is extracted.
    During validation/test, a center patch is used.
    """

    def __init__(
        self,
        split_csv,
        processed_dir="data/processed/mslesseg",
        patch_size=PATCH_SIZE,
        training=False,
    ):
        self.split_csv = Path(split_csv)
        self.processed_dir = Path(processed_dir)
        self.patch_size = tuple(patch_size)
        self.training = training

        self.df = pd.read_csv(self.split_csv)

        if len(self.df) == 0:
            raise ValueError(f"No samples found in {self.split_csv}")

    def __len__(self):
        return len(self.df)

    def _load_case(self, row):
        patient = str(row["patient"])
        timepoint = str(row["timepoint"])

        case_dir = self.processed_dir / patient / timepoint

        images = []

        for modality in MODALITIES:
            path = case_dir / f"{modality}.npy"

            if not path.exists():
                raise FileNotFoundError(f"Missing modality: {path}")

            image = np.load(path).astype(np.float32)
            images.append(image)

        mask_path = case_dir / "MASK.npy"

        if not mask_path.exists():
            raise FileNotFoundError(f"Missing mask: {mask_path}")

        mask = np.load(mask_path).astype(np.float32)

        image = np.stack(images, axis=0)
        mask = np.expand_dims(mask, axis=0)

        return image, mask

    def _get_random_start(self, shape):
        starts = []

        for size, patch in zip(shape, self.patch_size):
            if size < patch:
                raise ValueError(
                    f"Patch size {self.patch_size} is larger than "
                    f"image shape {shape}"
                )

            max_start = size - patch

            if max_start == 0:
                starts.append(0)
            else:
                starts.append(np.random.randint(0, max_start + 1))

        return starts

    def _get_center_start(self, shape):
        starts = []

        for size, patch in zip(shape, self.patch_size):
            if size < patch:
                raise ValueError(
                    f"Patch size {self.patch_size} is larger than "
                    f"image shape {shape}"
                )

            starts.append((size - patch) // 2)

        return starts

    def _crop_patch(self, image, mask):
        spatial_shape = image.shape[1:]

        if self.training:
            starts = self._get_random_start(spatial_shape)
        else:
            starts = self._get_center_start(spatial_shape)

        d, h, w = starts
        pd, ph, pw = self.patch_size

        image = image[
            :,
            d:d + pd,
            h:h + ph,
            w:w + pw,
        ]

        mask = mask[
            :,
            d:d + pd,
            h:h + ph,
            w:w + pw,
        ]

        return image, mask

    def __getitem__(self, index):
        row = self.df.iloc[index]

        image, mask = self._load_case(row)

        image, mask = self._crop_patch(image, mask)

        image = torch.from_numpy(image)
        mask = torch.from_numpy(mask)

        mask = (mask > 0).float()

        return {
            "image": image,
            "mask": mask,
            "patient": str(row["patient"]),
            "timepoint": str(row["timepoint"]),
        }

In [9]:
from torch.utils.data import DataLoader

PROCESSED_DIR = os.path.join(DATA_ROOT, "mslesseg_processed", "mslesseg")
SPLIT_DIR = os.path.join(DATA_ROOT, "splits")

train_dataset = MSLesSegDataset(
    split_csv=os.path.join(SPLIT_DIR, "train.csv"),
    processed_dir=PROCESSED_DIR,
    training=True,
)
val_dataset = MSLesSegDataset(
    split_csv=os.path.join(SPLIT_DIR, "validation.csv"),
    processed_dir=PROCESSED_DIR,
    training=False,
)

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))

sample = train_dataset[0]
print("Image shape:", tuple(sample["image"].shape))
print("Mask shape:", tuple(sample["mask"].shape))
print("Image NaN:", torch.isnan(sample["image"]).any().item())
print("Mask values:", torch.unique(sample["mask"]).tolist())

Train samples: 71
Val samples: 22
Image shape: (3, 112, 160, 128)
Mask shape: (1, 112, 160, 128)
Image NaN: False
Mask values: [0.0, 1.0]


In [10]:
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=2)
batch = next(iter(train_loader))
print("Image batch:", tuple(batch["image"].shape))
print("Mask batch:", tuple(batch["mask"].shape))
print("Patients in batch:", batch["patient"])

Image batch: (2, 3, 112, 160, 128)
Mask batch: (2, 1, 112, 160, 128)
Patients in batch: ['P29', 'P4']


In [11]:
model = build_model_a(in_channels=3, out_channels=1)
model.eval()

with torch.no_grad():
    logits = model(batch["image"])

print("Output logits shape:", tuple(logits.shape))  # expect (2, 1, 112, 160, 128)

loss_fn = FocalDiceLoss()
loss = loss_fn(logits, batch["mask"])
print("Loss (untrained, sanity check only):", loss.item())
print("Dice (untrained):", dice_score(logits, batch["mask"]))

Output logits shape: (2, 1, 112, 160, 128)
Loss (untrained, sanity check only): 0.6705520749092102
Dice (untrained): 0.017320765182375908


In [12]:
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = build_model_a(in_channels=3, out_channels=1).to(device)
loss_fn = FocalDiceLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=2)

model.train()
losses = []

start = time.time()
for step, batch in enumerate(train_loader):
    if step >= 20:   # smoke test only — 20 steps, not a real epoch
        break

    images = batch["image"].to(device)
    masks = batch["mask"].to(device)

    optimizer.zero_grad()
    logits = model(images)
    loss = loss_fn(logits, masks)
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    print(f"Step {step}: loss={loss.item():.4f}")

elapsed = time.time() - start
print(f"\n20 steps took {elapsed:.1f}s ({elapsed/20:.2f}s/step)")
print(f"71 train samples / batch_size=2 = {71//2} steps/epoch -> "
      f"~{(elapsed/20) * (71//2):.0f}s/epoch")
print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"GPU memory reserved:  {torch.cuda.memory_reserved()/1e9:.2f} GB")

Using device: cuda
Step 0: loss=0.6942
Step 1: loss=0.6848
Step 2: loss=0.6769
Step 3: loss=0.6646
Step 4: loss=0.6522
Step 5: loss=0.6455
Step 6: loss=0.6404
Step 7: loss=0.6249
Step 8: loss=0.6239
Step 9: loss=0.6347
Step 10: loss=0.6330
Step 11: loss=0.6286
Step 12: loss=0.6200
Step 13: loss=0.6127
Step 14: loss=0.6175
Step 15: loss=0.6142
Step 16: loss=0.6085
Step 17: loss=0.6126
Step 18: loss=0.5986
Step 19: loss=0.6180

20 steps took 16.0s (0.80s/step)
71 train samples / batch_size=2 = 35 steps/epoch -> ~28s/epoch
GPU memory allocated: 0.30 GB
GPU memory reserved:  1.92 GB


In [13]:
# training utilities
import random
import json
import csv
from pathlib import Path

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

@torch.no_grad()
def compute_metrics(logits, target, threshold=0.5, smooth=1e-5):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()

    preds_flat = preds.reshape(preds.shape[0], -1)
    target_flat = target.reshape(target.shape[0], -1)

    tp = (preds_flat * target_flat).sum(dim=1)
    fp = (preds_flat * (1 - target_flat)).sum(dim=1)
    fn = ((1 - preds_flat) * target_flat).sum(dim=1)

    dice = (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
    precision = (tp + smooth) / (tp + fp + smooth)
    recall = (tp + smooth) / (tp + fn + smooth)

    return {
        "dice": dice.mean().item(),
        "precision": precision.mean().item(),
        "recall": recall.mean().item(),
    }

@torch.no_grad()
def validate(model, loader, loss_fn, device):
    model.eval()
    total_loss, total_dice, total_prec, total_rec, n = 0.0, 0.0, 0.0, 0.0, 0

    for batch in loader:
        images = batch["image"].to(device)
        masks = batch["mask"].to(device)

        logits = model(images)
        loss = loss_fn(logits, masks)
        m = compute_metrics(logits, masks)

        total_loss += loss.item()
        total_dice += m["dice"]
        total_prec += m["precision"]
        total_rec += m["recall"]
        n += 1

    return {
        "val_loss": total_loss / n,
        "val_dice": total_dice / n,
        "val_precision": total_prec / n,
        "val_recall": total_rec / n,
    }

In [14]:
set_seed(42)

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
HISTORY_PATH = Path("/kaggle/working/model_a_history.csv")

NUM_EPOCHS = 150
PATIENCE = 20  # early stopping

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model_a(in_channels=3, out_channels=1).to(device)
loss_fn = FocalDiceLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=8
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, num_workers=2)

# Honesty/config record saved alongside every checkpoint
run_config = {
    "model": "Model A -- reproduction of Shifts-2.0 3D U-Net (single model, not 5-member ensemble)",
    "reason_for_reduction": "GPU budget constraint, documented explicitly",
    "in_channels": 3,
    "modalities": ["T1", "T2", "FLAIR"],
    "patch_size": [112, 160, 128],
    "loss": "FocalDiceLoss",
    "optimizer": "Adam",
    "lr": 1e-4,
    "seed": 42,
    "train_patients": 42,
    "val_patients": 11,
}
with open(CHECKPOINT_DIR / "run_config.json", "w") as f:
    json.dump(run_config, f, indent=2)

best_val_dice = -1.0
epochs_no_improve = 0
history = []

for epoch in range(NUM_EPOCHS):
    model.train()
    train_losses = []

    for batch in train_loader:
        images = batch["image"].to(device)
        masks = batch["mask"].to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = loss_fn(logits, masks)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    train_loss = sum(train_losses) / len(train_losses)
    val_metrics = validate(model, val_loader, loss_fn, device)
    scheduler.step(val_metrics["val_dice"])

    row = {"epoch": epoch, "train_loss": train_loss, **val_metrics,
           "lr": optimizer.param_groups[0]["lr"]}
    history.append(row)

    print(f"Epoch {epoch:3d} | train_loss={train_loss:.4f} | "
          f"val_loss={val_metrics['val_loss']:.4f} | "
          f"val_dice={val_metrics['val_dice']:.4f} | "
          f"val_prec={val_metrics['val_precision']:.4f} | "
          f"val_rec={val_metrics['val_recall']:.4f}")

    # Always save "last" so a disconnect never loses more than one epoch
    torch.save(model.state_dict(), CHECKPOINT_DIR / "model_a_last.pt")

    if val_metrics["val_dice"] > best_val_dice:
        best_val_dice = val_metrics["val_dice"]
        epochs_no_improve = 0
        torch.save(model.state_dict(), CHECKPOINT_DIR / "model_a_best.pt")
        print(f"  -> new best val_dice: {best_val_dice:.4f}, checkpoint saved")
    else:
        epochs_no_improve += 1

    # Save history every epoch, not just at the end
    with open(HISTORY_PATH, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        writer.writeheader()
        writer.writerows(history)

    if epochs_no_improve >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)")
        break

print(f"\nTraining done. Best val_dice: {best_val_dice:.4f}")
print(f"Checkpoints saved to: {CHECKPOINT_DIR}")

Epoch   0 | train_loss=0.6185 | val_loss=0.5838 | val_dice=0.0165 | val_prec=0.0085 | val_rec=0.5236
  -> new best val_dice: 0.0165, checkpoint saved
Epoch   1 | train_loss=0.5897 | val_loss=0.5758 | val_dice=0.0231 | val_prec=0.0120 | val_rec=0.5486
  -> new best val_dice: 0.0231, checkpoint saved
Epoch   2 | train_loss=0.5818 | val_loss=0.5713 | val_dice=0.0280 | val_prec=0.0147 | val_rec=0.4918
  -> new best val_dice: 0.0280, checkpoint saved
Epoch   3 | train_loss=0.5775 | val_loss=0.5679 | val_dice=0.0420 | val_prec=0.0226 | val_rec=0.4863
  -> new best val_dice: 0.0420, checkpoint saved
Epoch   4 | train_loss=0.5744 | val_loss=0.5659 | val_dice=0.0539 | val_prec=0.0299 | val_rec=0.4420
  -> new best val_dice: 0.0539, checkpoint saved
Epoch   5 | train_loss=0.5725 | val_loss=0.5642 | val_dice=0.0652 | val_prec=0.0375 | val_rec=0.3756
  -> new best val_dice: 0.0652, checkpoint saved
Epoch   6 | train_loss=0.5713 | val_loss=0.5632 | val_dice=0.0776 | val_prec=0.0463 | val_rec=0.3434

In [15]:
import shutil

shutil.move("/kaggle/working/checkpoints", "/kaggle/working/checkpoints_run1_uniform_sampling")
shutil.move("/kaggle/working/model_a_history.csv", "/kaggle/working/model_a_history_run1_uniform_sampling.csv")

print("Run 1 preserved.")

Run 1 preserved.


In [16]:
# foreground-biased patch sampling (new class, original file untouched)
class MSLesSegDatasetForegroundBiased(MSLesSegDataset):
    """
    Same as MSLesSegDataset, but training patches are biased toward
    lesion-containing regions instead of pure uniform random sampling.

    With probability fg_prob, the patch is centered (with random jitter)
    on a randomly chosen lesion voxel. Otherwise, falls back to the
    original uniform random sampling -- so the model still sees plenty
    of normal/background-only patches too.
    """

    def __init__(self, *args, fg_prob=0.75, **kwargs):
        super().__init__(*args, **kwargs)
        self.fg_prob = fg_prob

    def _get_foreground_start(self, mask, shape):
        lesion_voxels = np.argwhere(mask[0] > 0)  # mask is [1, D, H, W]

        if len(lesion_voxels) == 0:
            return None  # no lesion in this scan -- fall back to random

        center = lesion_voxels[np.random.randint(len(lesion_voxels))]

        starts = []
        for c, size, patch in zip(center, shape, self.patch_size):
            jitter = np.random.randint(-patch // 4, patch // 4 + 1)
            start = int(c) - patch // 2 + jitter
            start = max(0, min(start, size - patch))
            starts.append(start)

        return starts

    def _crop_patch(self, image, mask):
        spatial_shape = image.shape[1:]

        if self.training:
            if np.random.rand() < self.fg_prob:
                starts = self._get_foreground_start(mask, spatial_shape)
                if starts is None:
                    starts = self._get_random_start(spatial_shape)
            else:
                starts = self._get_random_start(spatial_shape)
        else:
            starts = self._get_center_start(spatial_shape)

        d, h, w = starts
        pd, ph, pw = self.patch_size

        image = image[:, d:d + pd, h:h + ph, w:w + pw]
        mask = mask[:, d:d + pd, h:h + ph, w:w + pw]

        return image, mask


train_dataset_fg = MSLesSegDatasetForegroundBiased(
    split_csv=os.path.join(SPLIT_DIR, "train.csv"),
    processed_dir=PROCESSED_DIR,
    training=True,
    fg_prob=0.75,
)

# quick sanity check: foreground-biased patches should contain far more lesion voxels on average
fg_counts = [train_dataset_fg[i]["mask"].sum().item() for i in range(10)]
uniform_counts = [train_dataset[i]["mask"].sum().item() for i in range(10)]
print("Foreground-biased lesion voxel counts (sample of 10):", fg_counts)
print("Uniform-random lesion voxel counts (sample of 10):    ", uniform_counts)

Foreground-biased lesion voxel counts (sample of 10): [9237.0, 17773.0, 6262.0, 2437.0, 2719.0, 4341.0, 2043.0, 9535.0, 7720.0, 1360.0]
Uniform-random lesion voxel counts (sample of 10):     [10473.0, 16626.0, 10794.0, 2965.0, 2715.0, 4285.0, 3177.0, 9495.0, 8234.0, 1735.0]


In [17]:
#Add a recall-favoring option: Tversky loss, a generalization of Dice that lets you penalize false negatives (missed lesions) more heavily than false positives
class TverskyLoss(nn.Module):
    """
    Generalized Dice: alpha weights false positives, beta weights false
    negatives. beta > alpha pushes the model toward higher recall --
    directly targeting the recall collapse seen in Run 1.
    """
    def __init__(self, alpha=0.3, beta=0.7, smooth=1e-5):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.smooth = smooth

    def forward(self, logits, target):
        probs = torch.sigmoid(logits)
        probs_flat = probs.reshape(probs.shape[0], -1)
        target_flat = target.reshape(target.shape[0], -1)

        tp = (probs_flat * target_flat).sum(dim=1)
        fp = (probs_flat * (1 - target_flat)).sum(dim=1)
        fn = ((1 - probs_flat) * target_flat).sum(dim=1)

        tversky = (tp + self.smooth) / (tp + self.alpha * fp + self.beta * fn + self.smooth)
        return (1 - tversky).mean()

In [18]:
# short pilot run (15 epochs, not a full commitment) combining both fixes
shutil.move("/kaggle/working/checkpoints", "/kaggle/working/checkpoints_run2_pilot") if os.path.exists("/kaggle/working/checkpoints") else None

set_seed(42)

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints_run2_pilot")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

model = build_model_a(in_channels=3, out_channels=1).to(device)
loss_fn = TverskyLoss(alpha=0.3, beta=0.7)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

train_loader_fg = DataLoader(train_dataset_fg, batch_size=4, shuffle=True, num_workers=2)

for epoch in range(15):
    model.train()
    train_losses = []
    for batch in train_loader_fg:
        images = batch["image"].to(device)
        masks = batch["mask"].to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = loss_fn(logits, masks)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    val_metrics = validate(model, val_loader, loss_fn, device)
    print(f"Epoch {epoch:2d} | train_loss={sum(train_losses)/len(train_losses):.4f} | "
          f"val_dice={val_metrics['val_dice']:.4f} | val_prec={val_metrics['val_precision']:.4f} | "
          f"val_rec={val_metrics['val_recall']:.4f}")

Epoch  0 | train_loss=0.9857 | val_dice=0.0530 | val_prec=0.0279 | val_rec=0.9403
Epoch  1 | train_loss=0.9829 | val_dice=0.0569 | val_prec=0.0300 | val_rec=0.9702
Epoch  2 | train_loss=0.9811 | val_dice=0.0637 | val_prec=0.0339 | val_rec=0.9766
Epoch  3 | train_loss=0.9800 | val_dice=0.0702 | val_prec=0.0376 | val_rec=0.9754
Epoch  4 | train_loss=0.9815 | val_dice=0.0732 | val_prec=0.0394 | val_rec=0.9775
Epoch  5 | train_loss=0.9798 | val_dice=0.0768 | val_prec=0.0415 | val_rec=0.9786
Epoch  6 | train_loss=0.9791 | val_dice=0.0850 | val_prec=0.0465 | val_rec=0.9767
Epoch  7 | train_loss=0.9784 | val_dice=0.0957 | val_prec=0.0527 | val_rec=0.9809
Epoch  8 | train_loss=0.9788 | val_dice=0.1008 | val_prec=0.0558 | val_rec=0.9778
Epoch  9 | train_loss=0.9782 | val_dice=0.1041 | val_prec=0.0581 | val_rec=0.9771
Epoch 10 | train_loss=0.9787 | val_dice=0.1114 | val_prec=0.0626 | val_rec=0.9774
Epoch 11 | train_loss=0.9781 | val_dice=0.1236 | val_prec=0.0701 | val_rec=0.9780
Epoch 12 | train

In [19]:
set_seed(42)

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints_run2_full")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
HISTORY_PATH = Path("/kaggle/working/model_a_history_run2.csv")

NUM_EPOCHS = 150
PATIENCE = 25

model = build_model_a(in_channels=3, out_channels=1).to(device)
loss_fn = TverskyLoss(alpha=0.3, beta=0.7)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=10
)

train_loader_fg = DataLoader(train_dataset_fg, batch_size=4, shuffle=True, num_workers=2)

run_config = {
    "model": "Model A -- reproduction of Shifts-2.0 3D U-Net (single model, not 5-member ensemble)",
    "run": "run2 -- foreground-biased sampling (fg_prob=0.75) + Tversky loss (alpha=0.3, beta=0.7)",
    "modalities": ["T1", "T2", "FLAIR"],
    "patch_size": [112, 160, 128],
    "seed": 42,
}
with open(CHECKPOINT_DIR / "run_config.json", "w") as f:
    json.dump(run_config, f, indent=2)

best_val_dice = -1.0
epochs_no_improve = 0
history = []

for epoch in range(NUM_EPOCHS):
    model.train()
    train_losses = []
    for batch in train_loader_fg:
        images = batch["image"].to(device)
        masks = batch["mask"].to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = loss_fn(logits, masks)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    train_loss = sum(train_losses) / len(train_losses)
    val_metrics = validate(model, val_loader, loss_fn, device)
    scheduler.step(val_metrics["val_dice"])

    row = {"epoch": epoch, "train_loss": train_loss, **val_metrics,
           "lr": optimizer.param_groups[0]["lr"]}
    history.append(row)

    print(f"Epoch {epoch:3d} | train_loss={train_loss:.4f} | "
          f"val_dice={val_metrics['val_dice']:.4f} | val_prec={val_metrics['val_precision']:.4f} | "
          f"val_rec={val_metrics['val_recall']:.4f} | lr={optimizer.param_groups[0]['lr']:.2e}")

    torch.save(model.state_dict(), CHECKPOINT_DIR / "model_a_last.pt")

    if val_metrics["val_dice"] > best_val_dice:
        best_val_dice = val_metrics["val_dice"]
        epochs_no_improve = 0
        torch.save(model.state_dict(), CHECKPOINT_DIR / "model_a_best.pt")
        print(f"  -> new best val_dice: {best_val_dice:.4f}, checkpoint saved")
    else:
        epochs_no_improve += 1

    with open(HISTORY_PATH, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        writer.writeheader()
        writer.writerows(history)

    if epochs_no_improve >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break

print(f"\nTraining done. Best val_dice: {best_val_dice:.4f}")

Epoch   0 | train_loss=0.9857 | val_dice=0.0530 | val_prec=0.0279 | val_rec=0.9406 | lr=1.00e-04
  -> new best val_dice: 0.0530, checkpoint saved
Epoch   1 | train_loss=0.9829 | val_dice=0.0569 | val_prec=0.0300 | val_rec=0.9702 | lr=1.00e-04
  -> new best val_dice: 0.0569, checkpoint saved
Epoch   2 | train_loss=0.9811 | val_dice=0.0637 | val_prec=0.0339 | val_rec=0.9767 | lr=1.00e-04
  -> new best val_dice: 0.0637, checkpoint saved
Epoch   3 | train_loss=0.9800 | val_dice=0.0701 | val_prec=0.0376 | val_rec=0.9754 | lr=1.00e-04
  -> new best val_dice: 0.0701, checkpoint saved
Epoch   4 | train_loss=0.9815 | val_dice=0.0732 | val_prec=0.0394 | val_rec=0.9775 | lr=1.00e-04
  -> new best val_dice: 0.0732, checkpoint saved
Epoch   5 | train_loss=0.9798 | val_dice=0.0768 | val_prec=0.0415 | val_rec=0.9785 | lr=1.00e-04
  -> new best val_dice: 0.0768, checkpoint saved
Epoch   6 | train_loss=0.9791 | val_dice=0.0851 | val_prec=0.0466 | val_rec=0.9768 | lr=1.00e-04
  -> new best val_dice: 0.0

In [20]:
model = build_model_a(in_channels=3, out_channels=1).to(device)
model.load_state_dict(torch.load("/kaggle/working/checkpoints_run2_full/model_a_best.pt"))
model.eval()

# Collect all val predictions once
all_probs = []
all_masks = []

with torch.no_grad():
    for batch in val_loader:
        images = batch["image"].to(device)
        masks = batch["mask"].to(device)
        probs = torch.sigmoid(model(images))
        all_probs.append(probs.cpu())
        all_masks.append(masks.cpu())

all_probs = torch.cat(all_probs, dim=0)
all_masks = torch.cat(all_masks, dim=0)

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]
results = []

for t in thresholds:
    preds = (all_probs > t).float()
    preds_flat = preds.reshape(preds.shape[0], -1)
    masks_flat = all_masks.reshape(all_masks.shape[0], -1)

    tp = (preds_flat * masks_flat).sum(dim=1)
    fp = (preds_flat * (1 - masks_flat)).sum(dim=1)
    fn = ((1 - preds_flat) * masks_flat).sum(dim=1)

    dice = ((2 * tp + 1e-5) / (2 * tp + fp + fn + 1e-5)).mean().item()
    prec = ((tp + 1e-5) / (tp + fp + 1e-5)).mean().item()
    rec = ((tp + 1e-5) / (tp + fn + 1e-5)).mean().item()

    results.append({"threshold": t, "dice": dice, "precision": prec, "recall": rec})
    print(f"threshold={t:.2f} | dice={dice:.4f} | precision={prec:.4f} | recall={rec:.4f}")

best = max(results, key=lambda r: r["dice"])
print(f"\nBest threshold: {best['threshold']} -> dice={best['dice']:.4f}")

threshold=0.30 | dice=0.0176 | precision=0.0089 | recall=1.0000
threshold=0.40 | dice=0.0308 | precision=0.0159 | recall=0.9757
threshold=0.50 | dice=0.2091 | precision=0.1263 | recall=0.9533
threshold=0.60 | dice=0.2291 | precision=0.1409 | recall=0.9454
threshold=0.70 | dice=0.2456 | precision=0.1535 | recall=0.9381
threshold=0.80 | dice=0.2630 | precision=0.1672 | recall=0.9294
threshold=0.90 | dice=0.2868 | precision=0.1869 | recall=0.9170
threshold=0.95 | dice=0.3073 | precision=0.2047 | recall=0.9063
threshold=0.99 | dice=0.3498 | precision=0.2452 | recall=0.8820

Best threshold: 0.99 -> dice=0.3498


In [21]:
thresholds_fine = [0.99, 0.995, 0.999, 0.9995, 0.9999, 0.99995, 0.99999]
results_fine = []

for t in thresholds_fine:
    preds = (all_probs > t).float()
    preds_flat = preds.reshape(preds.shape[0], -1)
    masks_flat = all_masks.reshape(all_masks.shape[0], -1)

    tp = (preds_flat * masks_flat).sum(dim=1)
    fp = (preds_flat * (1 - masks_flat)).sum(dim=1)
    fn = ((1 - preds_flat) * masks_flat).sum(dim=1)

    dice = ((2 * tp + 1e-5) / (2 * tp + fp + fn + 1e-5)).mean().item()
    prec = ((tp + 1e-5) / (tp + fp + 1e-5)).mean().item()
    rec = ((tp + 1e-5) / (tp + fn + 1e-5)).mean().item()

    results_fine.append({"threshold": t, "dice": dice, "precision": prec, "recall": rec})
    print(f"threshold={t:.5f} | dice={dice:.4f} | precision={prec:.4f} | recall={rec:.4f}")

best_fine = max(results_fine, key=lambda r: r["dice"])
print(f"\nBest fine threshold: {best_fine['threshold']} -> dice={best_fine['dice']:.4f}")

# Also worth seeing: what does the raw probability distribution actually look like?
print("\nProbability distribution over ALL voxels:")
print("  mean:", all_probs.mean().item())
print("  99th percentile:", torch.quantile(all_probs.flatten()[:2_000_000], 0.99).item())
print("  99.9th percentile:", torch.quantile(all_probs.flatten()[:2_000_000], 0.999).item())

print("\nProbability at true lesion voxels only:")
lesion_probs = all_probs[all_masks.bool()]
print("  mean:", lesion_probs.mean().item())
print("  median:", lesion_probs.median().item())

print("\nProbability at background voxels only:")
bg_probs = all_probs[~all_masks.bool()]
print("  mean:", bg_probs.mean().item())
print("  99th percentile:", torch.quantile(bg_probs[:2_000_000], 0.99).item())

threshold=0.99000 | dice=0.3498 | precision=0.2452 | recall=0.8820
threshold=0.99500 | dice=0.3669 | precision=0.2633 | recall=0.8696
threshold=0.99900 | dice=0.4047 | precision=0.3085 | recall=0.8346
threshold=0.99950 | dice=0.4198 | precision=0.3296 | recall=0.8173
threshold=0.99990 | dice=0.4499 | precision=0.3826 | recall=0.7702
threshold=0.99995 | dice=0.4594 | precision=0.4069 | recall=0.7455
threshold=0.99999 | dice=0.4681 | precision=0.4646 | recall=0.6750

Best fine threshold: 0.99999 -> dice=0.4681

Probability distribution over ALL voxels:
  mean: 0.4326380491256714
  99th percentile: 0.9999966621398926
  99.9th percentile: 1.0

Probability at true lesion voxels only:
  mean: 0.9719030261039734
  median: 0.9999980926513672

Probability at background voxels only:
  mean: 0.4277806282043457
  99th percentile: 0.999889612197876


In [22]:
flat_probs = all_probs.flatten()
flat_masks = all_masks.flatten()

sorted_probs, sort_idx = torch.sort(flat_probs, descending=True)
sorted_masks = flat_masks[sort_idx]

cum_tp = torch.cumsum(sorted_masks, dim=0)
total_lesion = flat_masks.sum()
k = torch.arange(1, len(sorted_masks) + 1, dtype=torch.float32)

cum_fp = k - cum_tp
cum_fn = total_lesion - cum_tp

dice_curve = (2 * cum_tp + 1e-5) / (2 * cum_tp + cum_fp + cum_fn + 1e-5)

best_idx = torch.argmax(dice_curve).item()
best_threshold = sorted_probs[best_idx].item()
best_dice = dice_curve[best_idx].item()
best_precision = (cum_tp[best_idx] / (cum_tp[best_idx] + cum_fp[best_idx])).item()
best_recall = (cum_tp[best_idx] / total_lesion).item()

print(f"Exact optimal threshold: {best_threshold:.8f}")
print(f"Dice at optimum:      {best_dice:.4f}")
print(f"Precision at optimum: {best_precision:.4f}")
print(f"Recall at optimum:    {best_recall:.4f}")

# sanity: confirm this is a true peak, not still climbing off the end
print(f"\nDice at last 5 points of curve: {dice_curve[-5:].tolist()}")

Exact optimal threshold: 0.99998236
Dice at optimum:      0.5533
Precision at optimum: 0.4759
Recall at optimum:    0.6608

Dice at last 5 points of curve: [0.01769600249826908, 0.01769600249826908, 0.01769600249826908, 0.017696000635623932, 0.017696000635623932]


In [23]:
# full-volume sliding-window evaluation
from monai.inferers import sliding_window_inference

model = build_model_a(in_channels=3, out_channels=1).to(device)
model.load_state_dict(torch.load("/kaggle/working/checkpoints_run2_full/model_a_best.pt"))
model.eval()

all_probs_full = []
all_masks_full = []

with torch.no_grad():
    for _, row in val_dataset.df.iterrows():
        image, mask = val_dataset._load_case(row)  # full volume, no cropping
        image_t = torch.from_numpy(image).unsqueeze(0).to(device)  # [1, 3, 182, 218, 182]

        logits = sliding_window_inference(
            inputs=image_t,
            roi_size=(112, 160, 128),
            sw_batch_size=1,
            predictor=model,
            overlap=0.5,
        )
        probs = torch.sigmoid(logits).cpu()

        all_probs_full.append(probs.squeeze(0))
        all_masks_full.append(torch.from_numpy(mask))

all_probs_full = torch.stack(all_probs_full)
all_masks_full = torch.stack(all_masks_full)
print("Full-volume shapes:", all_probs_full.shape, all_masks_full.shape)

Full-volume shapes: torch.Size([22, 1, 182, 218, 182]) torch.Size([22, 1, 182, 218, 182])


In [24]:
# find the real optimal threshold, on full volumes this time
flat_probs = all_probs_full.flatten()
flat_masks = all_masks_full.flatten()

sorted_probs, sort_idx = torch.sort(flat_probs, descending=True)
sorted_masks = flat_masks[sort_idx]

cum_tp = torch.cumsum(sorted_masks, dim=0)
total_lesion = flat_masks.sum()
k = torch.arange(1, len(sorted_masks) + 1, dtype=torch.float32)
cum_fp = k - cum_tp
cum_fn = total_lesion - cum_tp

dice_curve = (2 * cum_tp + 1e-5) / (2 * cum_tp + cum_fp + cum_fn + 1e-5)
best_idx = torch.argmax(dice_curve).item()

print(f"Full-volume optimal threshold: {sorted_probs[best_idx].item():.8f}")
print(f"Full-volume Dice at optimum:   {dice_curve[best_idx].item():.4f}")
print(f"Precision: {(cum_tp[best_idx] / (cum_tp[best_idx] + cum_fp[best_idx])).item():.4f}")
print(f"Recall:    {(cum_tp[best_idx] / total_lesion).item():.4f}")

Full-volume optimal threshold: 0.99999762
Full-volume Dice at optimum:   0.5626
Precision: 0.4863
Recall:    0.6673


In [25]:
# one-time test set evaluation
test_dataset = MSLesSegDataset(
    split_csv=os.path.join(SPLIT_DIR, "test.csv"),
    processed_dir=PROCESSED_DIR,
    training=False,
)
print("Test samples:", len(test_dataset))

all_probs_test = []
all_masks_test = []

with torch.no_grad():
    for _, row in test_dataset.df.iterrows():
        image, mask = test_dataset._load_case(row)
        image_t = torch.from_numpy(image).unsqueeze(0).to(device)

        logits = sliding_window_inference(
            inputs=image_t,
            roi_size=(112, 160, 128),
            sw_batch_size=1,
            predictor=model,
            overlap=0.5,
        )
        probs = torch.sigmoid(logits).cpu()

        all_probs_test.append(probs.squeeze(0))
        all_masks_test.append(torch.from_numpy(mask))

all_probs_test = torch.stack(all_probs_test)
all_masks_test = torch.stack(all_masks_test)

FINAL_THRESHOLD = 0.99999726  # locked from validation -- not re-tuned here

preds_test = (all_probs_test > FINAL_THRESHOLD).float()
preds_flat = preds_test.reshape(preds_test.shape[0], -1)
masks_flat = all_masks_test.reshape(all_masks_test.shape[0], -1)

tp = (preds_flat * masks_flat).sum(dim=1)
fp = (preds_flat * (1 - masks_flat)).sum(dim=1)
fn = ((1 - preds_flat) * masks_flat).sum(dim=1)

test_dice = ((2 * tp + 1e-5) / (2 * tp + fp + fn + 1e-5)).mean().item()
test_precision = ((tp + 1e-5) / (tp + fp + 1e-5)).mean().item()
test_recall = ((tp + 1e-5) / (tp + fn + 1e-5)).mean().item()

print(f"\nFINAL Model A test results (22 held-out patients, never touched before now):")
print(f"Test Dice:      {test_dice:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall:    {test_recall:.4f}")

Test samples: 22

FINAL Model A test results (22 held-out patients, never touched before now):
Test Dice:      0.3156
Test Precision: 0.2428
Test Recall:    0.7236


In [26]:
# save the full Model A summary (config + val + test results)
model_a_summary = {
    "model": "Model A -- reproduction of Shifts-2.0 3D U-Net (single model, not 5-member ensemble)",
    "modalities": ["T1", "T2", "FLAIR"],
    "patch_size": [112, 160, 128],
    "sampling": "foreground-biased (fg_prob=0.75)",
    "loss": "Tversky (alpha=0.3, beta=0.7)",
    "decision_threshold": FINAL_THRESHOLD,
    "threshold_selected_on": "validation set (22 scans), never re-tuned on test",
    "known_limitation": "probability outputs are poorly calibrated -- saturate near 0/1; revisit in reliability/calibration stage",
    "val_dice_full_volume": 0.5639,
    "val_precision": 0.4844,
    "val_recall": 0.6746,
    "test_dice_full_volume": test_dice,
    "test_precision": test_precision,
    "test_recall": test_recall,
    "checkpoint": "/kaggle/working/checkpoints_run2_full/model_a_best.pt",
}

with open("/kaggle/working/model_a_summary.json", "w") as f:
    json.dump(model_a_summary, f, indent=2)

print(json.dumps(model_a_summary, indent=2))

{
  "model": "Model A -- reproduction of Shifts-2.0 3D U-Net (single model, not 5-member ensemble)",
  "modalities": [
    "T1",
    "T2",
    "FLAIR"
  ],
  "patch_size": [
    112,
    160,
    128
  ],
  "sampling": "foreground-biased (fg_prob=0.75)",
  "loss": "Tversky (alpha=0.3, beta=0.7)",
  "decision_threshold": 0.99999726,
  "threshold_selected_on": "validation set (22 scans), never re-tuned on test",
  "known_limitation": "probability outputs are poorly calibrated -- saturate near 0/1; revisit in reliability/calibration stage",
  "val_dice_full_volume": 0.5639,
  "val_precision": 0.4844,
  "val_recall": 0.6746,
  "test_dice_full_volume": 0.3155549168586731,
  "test_precision": 0.24278336763381958,
  "test_recall": 0.7236312627792358,
  "checkpoint": "/kaggle/working/checkpoints_run2_full/model_a_best.pt"
}
